# 🐄💉 Cattle Disease Detection — Deep Learning for Veterinary Diagnostics

## Problem Statement
Automated detection of cattle diseases from images:
- **Foot-and-Mouth Disease (FMD)** — highly contagious viral disease
- **Lumpy Skin Disease (LSD)** — viral disease causing skin nodules
- **Healthy** — normal cattle for reference

## Model Architecture
**ConvNeXt-Tiny** — Modern hierarchical CNN, proven best for medical imaging on limited data
- **27.8M parameters** (75% frozen → 7M trainable)
- **Pre-trained on ImageNet** → transfer learning from natural images
- **Medical-grade augmentation** — preserve disease features while preventing overfitting

## Training Strategy
1. **Proper train/val/test split** (70/15/15) with stratification
2. **Class balancing** via weighted loss
3. **Strong regularization** — dropout, weight decay, label smoothing
4. **Medical augmentation** — moderate (don't distort lesions)
5. **5-crop TTA** at inference for reliable predictions
6. **Focal loss** — handles class imbalance better than CE

## Expected Performance
- **Target accuracy:** 95%+ on test set
- **Critical metric:** Per-class recall (sensitivity) — no missed diseases


In [ ]:
import os
import random
import shutil
from collections import Counter

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import transforms, models
from torchvision.datasets import ImageFolder

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, recall_score, f1_score

print("✅ All imports successful")


In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Device: {device}")


In [ ]:
# Dataset path
DATA_ROOT = "/kaggle/input/datasets/devang03mgr/cattle-diseases-datasets/Cows datasets"

print("📂 Dataset structure:")
for cls in sorted(os.listdir(DATA_ROOT)):
    cls_path = os.path.join(DATA_ROOT, cls)
    if os.path.isdir(cls_path):
        count = len([f for f in os.listdir(cls_path) if f.lower().endswith(('.jpg','.jpeg','.png'))])
        print(f"   {cls:20s}: {count:4d} images")

CLASS_NAMES = sorted([c for c in os.listdir(DATA_ROOT) if os.path.isdir(os.path.join(DATA_ROOT, c))])
NUM_CLASSES = len(CLASS_NAMES)
print(f"\n✅ Total classes: {NUM_CLASSES}")
print(f"   Classes: {CLASS_NAMES}")


In [ ]:
# Create clean train/val/test split with stratification
WORK_DIR = "/kaggle/working/disease_data"
TRAIN_DIR = os.path.join(WORK_DIR, "train")
VAL_DIR   = os.path.join(WORK_DIR, "val")
TEST_DIR  = os.path.join(WORK_DIR, "test")

# Clean up if exists
if os.path.exists(WORK_DIR):
    shutil.rmtree(WORK_DIR)

for d in [TRAIN_DIR, VAL_DIR, TEST_DIR]:
    os.makedirs(d, exist_ok=True)

# Split each class: 70% train, 15% val, 15% test
random.seed(SEED)

split_stats = []

for cls in CLASS_NAMES:
    src_dir = os.path.join(DATA_ROOT, cls)
    if not os.path.isdir(src_dir):
        continue
    
    # Get all images
    images = [f for f in os.listdir(src_dir) if f.lower().endswith(('.jpg','.jpeg','.png'))]
    random.shuffle(images)
    
    n = len(images)
    n_train = int(0.70 * n)
    n_val   = int(0.15 * n)
    
    # Split
    train_imgs = images[:n_train]
    val_imgs   = images[n_train:n_train + n_val]
    test_imgs  = images[n_train + n_val:]
    
    # Copy to respective folders
    for split_name, split_imgs, split_dir in [
        ("train", train_imgs, TRAIN_DIR),
        ("val", val_imgs, VAL_DIR),
        ("test", test_imgs, TEST_DIR)
    ]:
        dst = os.path.join(split_dir, cls)
        os.makedirs(dst, exist_ok=True)
        for img in split_imgs:
            shutil.copy(os.path.join(src_dir, img), os.path.join(dst, img))
    
    split_stats.append({
        "Class": cls,
        "Train": len(train_imgs),
        "Val": len(val_imgs),
        "Test": len(test_imgs),
        "Total": n
    })

df = pd.DataFrame(split_stats)
print("\n📊 Train/Val/Test Split:")
print(df.to_string(index=False))
print(f"\n✅ Total: Train={df['Train'].sum()} | Val={df['Val'].sum()} | Test={df['Test'].sum()}")


In [ ]:
# Medical imaging augmentation — preserve disease features, moderate augmentation
train_transforms = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(224, scale=(0.75, 1.0)),  # Not too aggressive
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),  # Small rotation (lesions shouldn't be distorted)
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),  # Moderate
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.2, scale=(0.02, 0.10))  # Small erasing
])

val_test_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# TTA: 5 crops for reliable medical predictions
def _norm():
    return transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])

tta_transforms = [
    transforms.Compose([transforms.Resize((224,224)), transforms.ToTensor(), _norm()]),
    transforms.Compose([transforms.Resize((224,224)), transforms.RandomHorizontalFlip(p=1.0), transforms.ToTensor(), _norm()]),
    transforms.Compose([transforms.Resize((256,256)), transforms.CenterCrop(224), transforms.ToTensor(), _norm()]),
    transforms.Compose([transforms.Resize((256,256)), transforms.CenterCrop(224), transforms.RandomHorizontalFlip(p=1.0), transforms.ToTensor(), _norm()]),
    transforms.Compose([transforms.Resize((288,288)), transforms.CenterCrop(224), transforms.ToTensor(), _norm()]),
]

print("✅ Medical-grade transforms ready")


In [ ]:
train_dataset = ImageFolder(TRAIN_DIR, transform=train_transforms)
val_dataset   = ImageFolder(VAL_DIR, transform=val_test_transforms)
test_dataset  = ImageFolder(TEST_DIR, transform=val_test_transforms)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

print(f"✅ DataLoaders ready")
print(f"   Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")
print(f"   Classes: {train_dataset.classes}")


In [ ]:
class DiseaseClassifier(nn.Module):
    """
    ConvNeXt-Tiny for cattle disease detection.
    """
    def __init__(self, num_classes):
        super().__init__()
        weights = models.ConvNeXt_Tiny_Weights.DEFAULT
        self.backbone = models.convnext_tiny(weights=weights)
        # Replace classifier head
        self.backbone.classifier[2] = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(768, 256),
            nn.GELU(),
            nn.Dropout(0.4),
            nn.Linear(256, num_classes)
        )
    
    def forward(self, x):
        return self.backbone(x)

model = DiseaseClassifier(num_classes=NUM_CLASSES).to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f"✅ Disease detection model created | Total params: {total_params/1e6:.1f}M")


In [ ]:
# Freeze bottom 75% of ConvNeXt for transfer learning
for name, p in model.backbone.named_parameters():
    if "classifier" in name:
        p.requires_grad = True  # Always train the head
        continue
    
    stage = -1
    if "features." in name:
        try:
            stage = int(name.split("features.")[1].split(".")[0])
        except (IndexError, ValueError):
            pass
    
    # Freeze stages 0-2 (indices 0-5), fine-tune stage 3 (indices 6-7)
    p.requires_grad = (stage < 0 or stage >= 6)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"✅ Freeze applied: {trainable/1e6:.2f}M / {total_params/1e6:.1f}M trainable ({100*trainable/total_params:.1f}%)")


In [ ]:
# Compute class weights for imbalanced medical data
labels_list = [lbl for _, lbl in train_dataset.samples]
counts = Counter(labels_list)
n_total = len(labels_list)

class_weights = torch.tensor(
    [n_total / counts[i] for i in range(NUM_CLASSES)],
    dtype=torch.float32
).to(device)

print("📊 Class distribution:")
for i, cls in enumerate(train_dataset.classes):
    print(f"   {cls:20s}: {counts[i]:4d} (weight: {class_weights[i]:.3f})")

# Focal Loss for better handling of hard examples in medical imaging
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0):
        super().__init__()
        self.alpha = alpha  # class weights
        self.gamma = gamma
    
    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, weight=self.alpha, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss
        return focal_loss.mean()

criterion = FocalLoss(alpha=class_weights, gamma=2.0)
print("\n✅ Focal Loss with class weights ready")


In [ ]:
EPOCHS = 50

# Discriminative LR: backbone top vs head
bb_params = [p for name, p in model.named_parameters() if "classifier" not in name and p.requires_grad]
head_params = [p for name, p in model.named_parameters() if "classifier" in name]

optimizer = optim.AdamW([
    {"params": bb_params,   "lr": 1e-5},
    {"params": head_params, "lr": 1e-3}
], weight_decay=1e-3)

scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-7)

print(f"✅ AdamW + CosineAnnealingLR | {EPOCHS} epochs | WD=1e-3")


In [ ]:
from torch.amp import GradScaler
scaler = GradScaler(device="cuda" if torch.cuda.is_available() else "cpu")
print("✅ Mixed Precision (AMP) enabled")


In [ ]:
def mixup(x, y, alpha=0.2):
    """Gentle MixUp for medical imaging — don't distort disease features too much"""
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(x.size(0))
    return lam * x + (1 - lam) * x[idx], y, y[idx], lam

print("✅ MixUp (α=0.2) ready")


In [ ]:
from torch.amp import autocast
_device_type = "cuda" if torch.cuda.is_available() else "cpu"

log_epochs = []
log_train_loss = []
log_val_acc = []
log_val_f1 = []
log_lr = []

PATIENCE = 10
P_MIXUP = 0.3

best_val_acc = 0.0
early_stop_counter = 0

for epoch in range(EPOCHS):
    # ──── TRAIN ────
    model.train()
    running_loss = 0.0

    for images, labels in train_loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad()

        with autocast(device_type=_device_type):
            # Apply MixUp with 30% probability
            if np.random.random() < P_MIXUP:
                mixed_images, y_a, y_b, lam = mixup(images, labels)
                outputs = model(mixed_images)
                loss = lam * criterion(outputs, y_a) + (1 - lam) * criterion(outputs, y_b)
            else:
                outputs = model(images)
                loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        running_loss += loss.item()

    scheduler.step()
    train_loss = running_loss / len(train_loader)

    # ──── VALIDATION ────
    model.eval()
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            
            with autocast(device_type=_device_type):
                outputs = model(images)
            
            all_preds.extend(outputs.argmax(1).cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    val_acc = (np.array(all_preds) == np.array(all_labels)).mean()
    val_f1 = f1_score(all_labels, all_preds, average='macro')

    log_epochs.append(epoch + 1)
    log_train_loss.append(train_loss)
    log_val_acc.append(val_acc)
    log_val_f1.append(val_f1)
    log_lr.append(optimizer.param_groups[0]["lr"])

    print(f"Epoch [{epoch+1:2d}/{EPOCHS}] | Loss: {train_loss:.4f} | "
          f"Val Acc: {val_acc:.4f} | Val F1: {val_f1:.4f} | LR: {optimizer.param_groups[0]['lr']:.2e}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "/kaggle/working/best_disease_model.pt")
        early_stop_counter = 0
        print("   ✅ Best model saved")
    else:
        early_stop_counter += 1
        print(f"   ⏳ {early_stop_counter}/{PATIENCE}")
        if early_stop_counter >= PATIENCE:
            print("🛑 Early stopping")
            break

print(f"\n🏆 Best validation accuracy: {best_val_acc:.4f}")

# Save training log
log_df = pd.DataFrame({
    "epoch": log_epochs,
    "train_loss": log_train_loss,
    "val_acc": log_val_acc,
    "val_f1": log_val_f1,
    "lr": log_lr
})
log_df.to_csv("/kaggle/working/training_log.csv", index=False)
print("✅ Training log saved to /kaggle/working/training_log.csv")


In [ ]:
FIG_DIR = "/kaggle/working/figures"
os.makedirs(FIG_DIR, exist_ok=True)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss
axes[0].plot(log_epochs, log_train_loss, marker='o', markersize=3, color='#e74c3c', linewidth=2)
axes[0].set_xlabel("Epoch", fontsize=12); axes[0].set_ylabel("Loss", fontsize=12)
axes[0].set_title("Training Loss", fontsize=14, fontweight='bold')
axes[0].grid(True, alpha=0.3)

# Accuracy
axes[1].plot(log_epochs, log_val_acc, marker='o', markersize=3, color='#27ae60', linewidth=2, label='Accuracy')
axes[1].axhline(best_val_acc, color='gray', ls='--', alpha=0.7, label=f'Best={best_val_acc:.3f}')
axes[1].set_xlabel("Epoch", fontsize=12); axes[1].set_ylabel("Accuracy", fontsize=12)
axes[1].set_title("Validation Accuracy", fontsize=14, fontweight='bold')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

# LR
axes[2].plot(log_epochs, log_lr, marker='o', markersize=3, color='#8e44ad', linewidth=2)
axes[2].set_xlabel("Epoch", fontsize=12); axes[2].set_ylabel("Learning Rate", fontsize=12)
axes[2].set_title("LR Schedule", fontsize=14, fontweight='bold')
axes[2].set_yscale('log'); axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f"{FIG_DIR}/training_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Training curves saved")


In [ ]:
model.load_state_dict(torch.load("/kaggle/working/best_disease_model.pt", map_location=device))
model.eval()
print("✅ Best model loaded")

# TTA on test set
from torch.amp import autocast
_device_type = "cuda" if torch.cuda.is_available() else "cpu"

all_labels = None
accum_probs = None

for t_idx, tfm in enumerate(tta_transforms):
    ds = ImageFolder(TEST_DIR, transform=tfm)
    loader = DataLoader(ds, batch_size=32, shuffle=False, num_workers=2)
    
    batch_probs = []
    batch_labels = []
    
    with torch.no_grad():
        for imgs, lbls in loader:
            with autocast(device_type=_device_type):
                outputs = model(imgs.to(device))
            batch_probs.append(torch.softmax(outputs.float(), dim=1).cpu())
            batch_labels.append(lbls)
    
    ep = torch.cat(batch_probs, 0)
    
    if accum_probs is None:
        accum_probs = ep
        all_labels = torch.cat(batch_labels, 0)
    else:
        accum_probs += ep
    
    print(f"   TTA crop {t_idx+1}/{len(tta_transforms)}")

all_preds = (accum_probs / len(tta_transforms)).argmax(1).numpy()
all_labels = all_labels.numpy()

test_acc = (all_preds == all_labels).mean()
test_f1 = f1_score(all_labels, all_preds, average='macro')

print(f"\n{'='*60}")
print(f"🏆 FINAL TEST RESULTS (5-crop TTA)")
print(f"{'='*60}")
print(f"   Accuracy:      {test_acc:.2%}")
print(f"   Macro F1:      {test_f1:.4f}")
print(f"{'='*60}")
print(f"\n📊 Detailed Classification Report:\n")
print(classification_report(all_labels, all_preds, target_names=train_dataset.classes, digits=4))


In [ ]:
FIG_DIR = "/kaggle/working/figures"
os.makedirs(FIG_DIR, exist_ok=True)

cm = confusion_matrix(all_labels, all_preds)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cm, xticklabels=train_dataset.classes, yticklabels=train_dataset.classes,
            cmap="Blues", annot=True, fmt="d", linewidths=1, square=True, 
            cbar_kws={"shrink": 0.8}, ax=ax, annot_kws={"fontsize": 14, "fontweight": "bold"})
ax.set_xlabel("Predicted", fontsize=13, fontweight='bold')
ax.set_ylabel("True", fontsize=13, fontweight='bold')
ax.set_title(f"Confusion Matrix | Test Accuracy: {test_acc:.2%}", fontsize=15, fontweight='bold', pad=20)
plt.xticks(fontsize=11)
plt.yticks(fontsize=11)
fig.tight_layout()
fig.savefig(f"{FIG_DIR}/confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Confusion matrix saved")


In [ ]:
FIG_DIR = "/kaggle/working/figures"
os.makedirs(FIG_DIR, exist_ok=True)

per_class_recall = recall_score(all_labels, all_preds, average=None)
per_class_precision = []
for i in range(NUM_CLASSES):
    tp = cm[i, i]
    fp = cm[:, i].sum() - tp
    per_class_precision.append(tp / (tp + fp) if (tp + fp) > 0 else 0)

classes = train_dataset.classes

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(classes))
width = 0.35

bars1 = ax.bar(x - width/2, per_class_recall, width, label='Recall (Sensitivity)', 
               color='#27ae60', edgecolor='black', linewidth=1)
bars2 = ax.bar(x + width/2, per_class_precision, width, label='Precision', 
               color='#3498db', edgecolor='black', linewidth=1)

ax.set_ylabel("Score", fontsize=13, fontweight='bold')
ax.set_title("Per-Class Performance — Disease Detection", fontsize=15, fontweight='bold', pad=20)
ax.set_xticks(x)
ax.set_xticklabels(classes, fontsize=11)
ax.set_ylim(0, 1.05)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, axis='y')

# Add value labels
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, height + 0.02,
                f"{height:.2%}", ha="center", va="bottom", fontsize=10, fontweight='bold')

fig.tight_layout()
fig.savefig(f"{FIG_DIR}/per_class_metrics.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Per-class metrics plot saved")


In [ ]:
print("\n" + "="*70)
print("📈 FINAL SUMMARY — CATTLE DISEASE DETECTION")
print("="*70)
print(f"Model:           ConvNeXt-Tiny (Medical Transfer Learning)")
print(f"Total Params:    {total_params/1e6:.1f}M")
print(f"Trainable:       {trainable/1e6:.2f}M ({100*trainable/total_params:.1f}%)")
print(f"Dataset:         {len(train_dataset)+len(val_dataset)+len(test_dataset)} images")
print(f"Classes:         {NUM_CLASSES} ({', '.join(train_dataset.classes)})")
print(f"Epochs Trained:  {len(log_epochs)}")
print(f"Best Val Acc:    {best_val_acc:.2%}")
print(f"Test Acc (TTA):  {test_acc:.2%}")
print(f"Macro F1:        {test_f1:.4f}")
print("="*70)
print(f"\n🔬 Per-Class Recall (Sensitivity):")
for i, cls in enumerate(classes):
    status = "✅" if per_class_recall[i] >= 0.90 else "⚠️" if per_class_recall[i] >= 0.80 else "❌"
    print(f"   {status} {cls:20s}: {per_class_recall[i]:.2%}")
print("="*70)
print(f"\n✅ Artifacts saved to: {FIG_DIR}/")
print(f"✅ Model saved to: /kaggle/working/best_disease_model.pt")


In [ ]:
# Save final results summary as CSV
results_df = pd.DataFrame({
    "metric": ["test_accuracy", "macro_f1", "best_val_accuracy"],
    "value": [test_acc, test_f1, best_val_acc]
})
results_df.to_csv("/kaggle/working/final_results.csv", index=False)

# Save per-class metrics
class_metrics_df = pd.DataFrame({
    "class": list(train_dataset.classes),
    "recall": list(per_class_recall),
    "precision": list(per_class_precision)
})
class_metrics_df.to_csv("/kaggle/working/per_class_metrics.csv", index=False)

print("✅ All outputs saved to /kaggle/working/:")
print("   📊 training_log.csv")
print("   📊 final_results.csv")
print("   📊 per_class_metrics.csv")
print("   🖼️  figures/training_curves.png")
print("   🖼️  figures/confusion_matrix.png")
print("   🖼️  figures/per_class_metrics.png")
print("   🧠 best_disease_model.pt")
